email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
from dotenv import load_dotenv

load_dotenv('E:/Courses/Intro to Langchain/lca-lc-foundations-main/example.env')

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="draft 1")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Sure—draft for what? A first draft can be an email, memo, report, proposal, blog post, etc. To tailor it, please share:

- Type of draft (email, memo, report, proposal, blog post, etc.)
- Topic or purpose
- Audience
- Key points or sections to include
- Desired length or word count
- Tone (formal, neutral, casual)
- any deadlines or formatting requirements

If you’re not sure, I can start with a generic 1-page business email draft. Just say “start generic” or specify the details above.


In [10]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Sure—draft for what? A first draft can be an email, memo, report, proposal, blog post, etc. To tailor it, please share:

- Type of draft (email, memo, report, proposal, blog post, etc.)
- Topic or purpose
- Audience
- Key points or sections to include
- Desired length or word count
- Tone (formal, neutral, casual)
- any deadlines or formatting requirements

If you’re not sure, I can start with a generic 1-page business email draft. Just say “start generic” or specify the details above.


In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='draft 1', additional_kwargs={}, response_metadata={}, id='986250e0-0561-4cdc-a427-7702514db934'),
              AIMessage(content='Sure—draft for what? A first draft can be an email, memo, report, proposal, blog post, etc. To tailor it, please share:\n\n- Type of draft (email, memo, report, proposal, blog post, etc.)\n- Topic or purpose\n- Audience\n- Key points or sections to include\n- Desired length or word count\n- Tone (formal, neutral, casual)\n- any deadlines or formatting requirements\n\nIf you’re not sure, I can start with a generic 1-page business email draft. Just say “start generic” or specify the details above.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1025, 'prompt_tokens': 147, 'total_tokens': 1172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, '